# Pair-Aware BERTimbau Base — RECLin-PT (Colab T4)

Treina a estratégia **Pair-Aware** de extração de relação em 3 classes (`negation_of`, `associated_with`, `no_relation`) sobre as partições do SemClinBr usando o **BERTimbau Base** (`neuralmind/bert-base-portuguese-cased`), encoder **geral** do português brasileiro. É a variante de encoder do Pair-Aware linear de `reclin_pair_aware.ipynb` (`pucpr/biobertpt-all`), que **não** é alterado: mesma estratégia `pair_aware`, mesma cabeça, mesmos dados, partições, hiperparâmetros e seed; **só muda o encoder** (e, com ele, a revisão dos pesos, o `config_id` e os nomes das pastas). O BERTimbau é uma **variante fora do pré-registro** da fase (que fixa `pucpr/biobertpt-all`): não recebe nome oficial.

A Pair-Aware (protocolo, D10) é o baseline com uma cabeça que lê o par de entidades: `[h_cls ; h_[E1] ; h_[E2]] → Dropout → Linear(3d, 3)`, **sem MLP de fusão**. Treina e avalia em **todos** os pares candidatos elegíveis: **sem filtro lexical** e **sem usar o léxico** de nenhuma forma (nem como feature). Por isso as métricas são do **DEV completo** e comparáveis com as do Baseline e do Two-Stage.

Configuração do protocolo (herdada do Pair-Aware linear com BioBERTpt-all): seed 42, **3 épocas, `batch_size=64`**, sem parada antecipada; a época escolhida é a de maior **macro-F1 do DEV** (empate → a primeira). Nenhum limiar é escolhido neste notebook.

Métricas no **DEV**: Macro/Micro/Weighted-F1, **MCC**, F1 por classe, matriz de confusão e curvas de treino/validação. **Este notebook não avalia o TEST**: a avaliação do TEST é uma etapa separada, feita uma única vez, depois de congelar a seleção (protocolo `docs/protocolo/fase-biobertpt-seed42.md`).

Antes de rodar: **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)** e cadastre os secrets `GITHUB_PAT` e `HF_TOKEN` (ícone de chave 🔑, com **Notebook access** ligado).

O treino é feito no disco local e **espelhado a cada checkpoint no Google Drive** (`MyDrive/RECLin-PT/execucoes/`); se o runtime cair, **rode a seção 7 de novo**: ela **retoma automaticamente** da execução interrompida compatível.

Envios para fora (GitHub e Hugging Face) só acontecem nas seções 10 e 11, e só depois de você mudar a confirmação explícita para `True`.


## 1. Clonar o repositório

In [ ]:
import os, shutil, subprocess, sys, tempfile
from pathlib import Path
from google.colab import userdata

try:
    userdata.get('GITHUB_PAT')               # secret do Colab, nunca hard-coded (o valor não é guardado nem impresso)
except Exception as erro:
    raise SystemExit('Secret GITHUB_PAT indisponível (%s). Colab: ícone de chave → "+ Add new secret" → nome '
                     'GITHUB_PAT → cole o token → ligue "Notebook access" → rode esta célula de novo.' % type(erro).__name__)
REPO_NAME = 'RECLin-PT'                       # nome do repo no GitHub
REPO = f'/content/{REPO_NAME}'
GIT_USER_NAME = 'angeloalsf'
GIT_USER_EMAIL = 'angeloalsf@gmail.com'

# O token NÃO entra na URL (ficaria gravado em .git/config): vai por GIT_ASKPASS (script 700, apagado ao
# terminar), só no ambiente do processo do git.
url = f'https://github.com/angeloalsf/{REPO_NAME}.git'

def git(*args, check=True):
    pasta = tempfile.mkdtemp(prefix='askpass-'); os.chmod(pasta, 0o700)
    script = os.path.join(pasta, 'askpass.sh')
    with open(script, 'w') as f:
        f.write('#!/bin/sh\ncase "$1" in *sername*) printf "x-access-token";; *) printf "%s" "$RECLIN_GIT_TOKEN";; esac\n')
    os.chmod(script, 0o700)
    env = {**os.environ, 'GIT_ASKPASS': script, 'GIT_TERMINAL_PROMPT': '0', 'RECLIN_GIT_TOKEN': userdata.get('GITHUB_PAT')}
    try:
        return subprocess.run(['git', '-c', 'credential.helper=', *args], env=env, check=check)
    finally:
        shutil.rmtree(pasta, ignore_errors=True)

if not os.path.isdir(REPO):
    git('clone', url, REPO)
else:
    print('Repo já clonado; fazendo pull --rebase para integrar mudanças remotas.')
    git('-C', REPO, 'pull', '--rebase', '--autostash', check=False)

git('-C', REPO, 'config', 'user.name', GIT_USER_NAME)
git('-C', REPO, 'config', 'user.email', GIT_USER_EMAIL)
git('-C', REPO, 'remote', 'set-url', 'origin', url)

# Código novo do projeto: fica em REPO/codigo. Registra o commit clonado (HEAD) na proveniência de cada execução.
BRANCH = subprocess.run(['git', '-C', REPO, 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO}/codigo')
from reclin.experimento import apresentacao, codigo_colab, operacoes, plano, raiz, segredos
INFO = codigo_colab.clonar_codigo(f'angeloalsf/{REPO_NAME}', BRANCH, REPO)
SHA_CODIGO = INFO['sha']
CODIGO = Path(INFO['codigo'])
raiz.ativar(CODIGO)                          # põe codigo/ no sys.path (devolve o ambiente de subprocessos, não o caminho)
print('Repo pronto em', REPO, f'| branch {BRANCH} | commit {SHA_CODIGO}')

## 2. Verificar a GPU (T4)

In [ ]:
!nvidia-smi
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execução → GPU (T4).'

## 3. Montar o Google Drive (checkpoints)

Execuções em `MyDrive/RECLin-PT/execucoes/<nome>/` — **pasta própria** de cada execução (o nome inclui a configuração). Dentro: `checkpoints/melhor_modelo/` (pesos da melhor época) e `checkpoints/ultimo.pt` (estado de retomada). O treino roda no disco local e o Drive recebe uma cópia verificada (SHA-256).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RAIZ_DRIVE = Path(f'/content/drive/MyDrive/{REPO_NAME}')
RAIZ_DRIVE.mkdir(parents=True, exist_ok=True)
RAIZ_LOCAL = Path('/content/reclin_resultados/execucoes')       # disco local do Colab
RAIZ_PESOS = Path('/content/reclin_pesos')
print('Execuções espelhadas em:', RAIZ_DRIVE)
existe = any(e['estado'] in ('interrompida', 'iniciada') for e in plano.execucoes(RAIZ_LOCAL, RAIZ_DRIVE))
print('execução interrompida existente?', existe, '(se True, o treino vai RETOMAR de onde parou)')

## 4. Instalar dependências

**Só esta célula instala dependências** (nada é instalado ao abrir o notebook nem em outras seções). Fonte única: `codigo/requirements.txt` do repo — a mesma lista usada fora do Colab, com as versões **fixadas com `==`**.

O `pip` **não reinstala** o `torch` enquanto a imagem do Colab trouxer a versão fixada (`torch==2.11.0`; o `+cu128` da imagem é ignorado na comparação pela PEP 440), e o mesmo vale para os demais pins que já batem com a imagem, então esta célula é rápida.

⚠️ Isso vale **enquanto o Colab ficar no 2.11.0**. Se a imagem subir e esta célula começar a demorar (baixa o `torch` do PyPI, vários GB), recapture as versões e suba o pin, nunca volte para `>=`. Nunca escreva `torch==2.11.0+cu128`. Se o pip trocar um pacote já importado (por exemplo `numpy`), o Colab pede para **reiniciar a sessão**: reinicie e rode tudo de novo, do início.

In [ ]:
!pip install -q -r $REPO/codigo/requirements.txt

## 5. Conferir os splits e os pesos

As partições (`codigo/dados/particoes-v2/*.jsonl`, seed 42, nível de documento) são **versionadas e congeladas** no repo — **as mesmas de todas as estratégias**. Esta célula só **confere** (arquivos e SHA-256 contra o MANIFEST); **não baixa nem reconstrói** dados.

**Só a segunda célula desta seção baixa os pesos** (nenhuma outra seção acessa o Hub para baixar; o treino recusa se eles faltarem). Os pesos do encoder são baixados do Hub na **revisão fixada** (`REVISAO_PESOS`), com conferência dos hashes dos arquivos; o commit usado fica registrado e congelado para a fase.

In [ ]:
ENCODER = 'bertimbau'                       # chave em reclin.config.ENCODERS
REPO_PESOS = 'neuralmind/bert-base-portuguese-cased'
NOME_PESOS = 'bert-base-portuguese-cased'   # pasta dos pesos: bert-base-portuguese-cased@<revisão> (nunca a de biobertpt-all)
REVISAO_PESOS = '94d69c95f98f7d5b2a8700c420230ae10def0baa'   # commit de neuralmind/bert-base-portuguese-cased (SHA congelado; confira no Hub antes do 1º treino)

def _espec():
    from reclin.estrategias import pair_aware      # importada na hora (depois da instalação da seção 4)
    from reclin.config import ENCODERS
    assert ENCODERS[ENCODER] == REPO_PESOS, 'ENCODER e REPO_PESOS divergem'
    # todos os pares, sem filtro lexical e sem léxico: a especificação não carrega nada além da configuração
    return operacoes.Especificacao(estrategia=pair_aware.NOME, script='treinar.py', config_cls=pair_aware.ConfigPairAware,
                                   estrategia_config=pair_aware.montagem().registro, encoder=ENCODER)

def _ctx(**kw):
    return apresentacao.contexto(CODIGO, RAIZ_LOCAL, RAIZ_DRIVE, RAIZ_PESOS, REVISAO_PESOS, _espec(),
                                 repositorio_pesos=REPO_PESOS, nome_pesos=NOME_PESOS, **kw)

def _ctx_exec(restaurar=False, **kw):
    # contexto da execução mais recente (local ou Drive); com restaurar=True traz do Drive para o disco local
    try:
        return _ctx(execucao=apresentacao.execucao_atual(_ctx(), restaurar=restaurar), **kw)
    except operacoes.ErroOperacao as erro:
        raise SystemExit(f'ERRO: {erro}') from None

splits = os.path.join(REPO, 'codigo', 'dados', 'particoes-v2')
need = ['train.jsonl', 'dev.jsonl', 'test.jsonl', 'MANIFEST.json']
have = all(os.path.isfile(os.path.join(splits, f)) for f in need)

if not have:
    raise FileNotFoundError('Partições não encontradas em codigo/dados/particoes-v2/. Elas são versionadas e '
                            'congeladas; este notebook não baixa nem reconstrói dados. Confira a branch clonada.')

PART = apresentacao.conferir_particoes(_ctx(), sha_codigo=SHA_CODIGO)
assert PART and PART['ok'], 'Partições não conferem com o MANIFEST (veja ERRO acima).'

In [ ]:
apresentacao.avisar_revisao_pesos(_ctx())
RES_PESOS = operacoes.executar(operacoes.OP_PESOS, _ctx())
assert RES_PESOS, 'Pesos indisponíveis (veja ERRO acima).'

## 6. Token do Hugging Face Hub (Colab Secrets)

Prepara o **backup** da execução num repositório **privado** do HF Hub (`angeloalsf/reclin-pt-pairaware-bertimbau-seed42`). É camada **extra**: o Drive continua sendo o destino principal. O envio é feito **ao final**, na seção 11, e só com a sua confirmação explícita (o `treinar.py` atual não envia nada durante o treino).

O token nunca é colado em texto plano aqui; vem dos **Secrets** do Colab, e **não é exportado** para o ambiente nesta célula.

1. Crie um Access Token do tipo **Write** em <https://huggingface.co/settings/tokens>
2. No Colab: ícone de **chave** no painel esquerdo → **+ Add new secret** → nome `HF_TOKEN` → cole o valor → ligue **Notebook access**
3. Rode a célula abaixo

Sem o secret o treino roda igual; só o backup extra fica de fora.

In [ ]:
from google.colab import userdata

HF_USER = 'angeloalsf'
MODELO = 'bertimbau'

# A semente é a da fase, vem da constante do protocolo (não de nome de pasta).
SEED, _ = apresentacao.semente_da_fase()
HF_BACKUP_REPO = f'{HF_USER}/reclin-pt-pairaware-{MODELO}-seed{SEED}'

try:
    cred = segredos.obter(segredos.HUGGING_FACE)
except Exception as erro:
    cred = None
    print('Secret HF_TOKEN indisponivel (%s).' % type(erro).__name__)

HF_OK = False
if cred:
    try:
        quem = segredos.verificar_escrita_hf(cred)        # consulta somente leitura (whoami); nada é enviado
        HF_OK = True
        print('HF_TOKEN carregado dos Secrets do Colab; token de escrita valido (usuario %s).' % quem['usuario'])
        print('(o valor nunca e impresso, nem exportado para o ambiente nesta celula).')
        print('Backup (secao 11) ->', HF_BACKUP_REPO, '(repositorio PRIVADO)')
        print('Semente da fase:', SEED)
    except Exception as erro:
        print('HF_TOKEN presente, mas nao validou como token de escrita (%s).' % type(erro).__name__)
cred = None
if not HF_OK:
    print('BACKUP NO HF HUB DESLIGADO nesta execucao.')
    print('  1. Crie um token WRITE em https://huggingface.co/settings/tokens')
    print('  2. Colab: icone de chave (Secrets) no painel esquerdo >')
    print('     "+ Add new secret" > nome HF_TOKEN > cole o valor >')
    print('     ligue "Notebook access"')
    print('  3. Rode esta celula de novo.')
    print('  O treino roda normalmente; so o backup extra fica de fora.')

## 7. Treinar a Pair-Aware BERTimbau Base (com retomada)

Encoder `neuralmind/bert-base-portuguese-cased`, marcadores `[E1] [/E1] [E2] [/E2]`, 3 classes, `class_weight=balanced`, cabeça linear `[h_cls ; h_E1 ; h_E2] → Dropout → Linear` (**sem MLP**; `head_dropout=None` usa o dropout do encoder). **Todos** os pares candidatos entram no treino e no DEV (sem filtro lexical, sem léxico); **3 épocas**, `batch_size=64`, sem parada antecipada. Melhor época pelo macro-F1 do **DEV completo**; o **test não é usado aqui**.

Antes do treino é medido quantos exemplos ficam sem `[E1]` ou `[E2]` depois da tokenização e caem no fallback para `h_cls` (`descricao.marcadores_ausentes`, em `treino.json`).

A configuração e a cabeça entram na chave da execução, e os checkpoints são da estratégia `pair_aware`: execuções de outra estratégia — e as da Pair-Aware antiga (restrita, com MLP) — nunca são retomadas nem reaproveitadas aqui.

**Hiperparâmetros do protocolo.** A célula escolhe sozinha o que fazer: **treino novo** se não há execução do protocolo, **retomada** se há uma interrompida compatível (ela pula as épocas já feitas) e, se o protocolo já foi concluído, **não treina de novo** (nada é sobrescrito). Se o runtime cair, **rode esta célula de novo**.

> Mudar qualquer valor de `PARAMETROS` torna a execução **exploratória** (fora da fase científica, não elegível ao TEST), e só ela roda com `NOVA_EXECUCAO = True`. **Retomar não é iniciar outra tentativa:** a retomada só aceita execução **compatível** (mesma configuração, dados, pesos e identidade do código, conferidos antes de reutilizar o checkpoint); se não houver, a célula para e mostra o que difere. Uma execução exploratória tem outra configuração, logo outra pasta, e nunca reaproveita checkpoints da oficial. Com uma execução interrompida compatível, `NOVA_EXECUCAO = True` exige `MOTIVO`.
>
> ⚠️ **`max_gap` = 25 (era 20).** Mudou após a análise de sensibilidade em `analysis/max_gap/`: a janela de 20 caracteres descartava 1.324 relações anotadas antes de virarem candidato.

In [ ]:
PARAMETROS = {                 # valores do protocolo (D10): 3 épocas, lote 64, cabeça linear sem MLP
    'max_gap': 25, 'ctx_chars': 128, 'max_length': 128, 'epochs': 3, 'batch_size': 64, 'lr': 2e-5,
    'weight_decay': 0.01, 'warmup_ratio': 0.1, 'max_grad_norm': 1.0, 'class_weight': 'balanced',
    'head_dropout': None,          # None = dropout do encoder
}
NOVA_EXECUCAO = False          # True só para outra tentativa ou parâmetros diferentes do protocolo (exploratória)
MOTIVO = ''                    # opcional (vazio grava "sem motivo informado")

ctx = _ctx(parametros=PARAMETROS, motivo=MOTIVO.strip() or 'sem motivo informado')
protocolo = operacoes.config_da_operacao(ctx.espec, PARAMETROS) == ctx.espec.config_protocolo()
rec = apresentacao.recomendar_ramo(ctx)
if NOVA_EXECUCAO and rec['recomendada'] == operacoes.OP_RETOMAR and not MOTIVO.strip():
    raise SystemExit(f"Há execução interrompida COMPATÍVEL ({', '.join(rec['compativeis'])}). Para RETOMAR, ponha "
                     "NOVA_EXECUCAO = False. Para abandoná-la e iniciar OUTRA tentativa, preencha MOTIVO.")
OPERACAO = operacoes.OP_NOVA if NOVA_EXECUCAO else rec['recomendada']
print('\nOperação:', OPERACAO, '| natureza:',
      'PARÂMETROS DO PROTOCOLO, encoder bertimbau (variante fora do pré-registro; DEV apenas)' if protocolo else 'EXPLORATÓRIA (fora da fase científica, não elegível ao TEST)')
if OPERACAO == operacoes.OP_NOVA and not NOVA_EXECUCAO:
    print('\nNada foi treinado:', rec['motivo'])
    print('Para outra tentativa ou parâmetros diferentes do protocolo (exploratória), ponha NOVA_EXECUCAO = True.')
else:
    QUADRO = apresentacao.quadro_pre_treino(ctx, OPERACAO, sha_codigo=SHA_CODIGO, repositorio=f'angeloalsf/{REPO_NAME}')
    try:
        apresentacao.exigir_quadro(QUADRO, ctx, OPERACAO)
    except operacoes.ErroOperacao as erro:
        raise SystemExit(f'ERRO: {erro}') from None
    RESULTADO_TREINO = operacoes.executar(OPERACAO, ctx)

## 8. Resultados (métricas + matriz de confusão)

Da execução mais recente (ela fica **selecionada** para as seções 10 e 11; o nome e a natureza — protocolo ou exploratória — são impressos), **só DEV**. O TEST não é lido nem avaliado aqui.

In [ ]:
from reclin.experimento import proveniencia

CTX = _ctx_exec(restaurar=True)
EXECUCAO_SELECIONADA = CTX.execucao            # as seções 10 e 11 enviam SÓ esta execução
_reg = proveniencia.ler(apresentacao.pasta_local_da_execucao(CTX))
_base = CTX.espec.config_protocolo().como_dict()
NATUREZA = 'PARÂMETROS DO PROTOCOLO, encoder bertimbau (variante fora do pré-registro; DEV apenas)' if all(_reg['parametros_de_treinamento'].get(k) == _base[k] for k in CTX.espec.permitidos) else 'EXPLORATÓRIA'
print(f"Execução selecionada: {EXECUCAO_SELECIONADA} | natureza: {NATUREZA} | estado: {_reg['estado']}")
operacoes.executar(operacoes.OP_DEV, CTX)

### 8.1 Checkpoint selecionado: métricas por classe e proveniência

Critério de seleção (global, o do protocolo): **maior macro-F1 do DEV completo** (as 3 classes pesam igual; `negation_of` sozinho não decide), empate → a primeira época. As métricas abaixo vêm do `predicoes_dev.json` do checkpoint selecionado (o DEV completo, 19.064 candidatos; o que a Pair-Aware não pontua vira `no_relation`).

Por época, o `treino.json` guarda apenas `train_loss`, `dev_loss`, `dev_macro_f1` e `dev_negation_of_f1`. P/R por classe a cada época **não** são gravados (exigiria mudar o laço de treino compartilhado); só o checkpoint selecionado tem o detalhamento completo.

In [ ]:
from reclin.experimento import comparacao_dev
from reclin.execucao import predicoes as _pred

import json
PASTA = apresentacao.pasta_local_da_execucao(CTX)
_trn = json.load(open(PASTA / 'treino.json', encoding='utf-8'))
_cfg = json.load(open(PASTA / 'config.json', encoding='utf-8'))
_dev = _pred.ler(PASTA / 'predicoes_dev.json')
_l = comparacao_dev.linha(EXECUCAO_SELECIONADA, _dev)

_ec = (_reg.get('chave_configuracao') or {}).get('estrategia_config') or {}
print('Critério de seleção :', _ec.get('selecao_melhor_epoca') or 'dev_macro_f1 no DEV completo (protocolo)')
print('Melhor época        :', _trn['melhor_epoca'], '| dev_macro_f1 =', round(_trn['melhor_dev_macro_f1'], 4))
print('\nPor época (DEV completo):')
for e in _trn['dev_history']:
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in e.items()})
print(f"\nDEV completo (n = {_l['n']}) no checkpoint selecionado:  macro-F1 {_l['macro_f1']:.4f} | acurácia {_l['acuracia']:.4f} | MCC {_l['mcc']:.4f}")
for r in comparacao_dev.FOCO:
    print(f"  {r:16s} P {_l[r + '.precisao']:.4f}  R {_l[r + '.recall']:.4f}  F1 {_l[r + '.f1']:.4f}  suporte {int(_l[r + '.suporte'])}")
print('Matriz de confusão (linhas = verdadeiro, colunas = predito; negation_of, associated_with, no_relation):')
for lin in _l['matriz_confusao']: print('  ', lin)
_prov = _cfg.get('proveniencia_pesos') or {}
print('\nProveniência: seed', _reg['seed'], '| revisão dos pesos', _prov.get('commit') or _reg['pesos_pretreinados'].get('commit'),
      '| hash do manifesto dos pesos', _prov.get('hash_manifesto') or _reg['pesos_pretreinados'].get('hash_manifesto'))
print('  código:', {k: _reg['codigo'].get(k) for k in ('origem', 'revisao')}, '| identidade:', _reg['codigo'].get('identidade_global'))
print('  sha256 do config.json:', _reg.get('sha256_config'), '| natureza:', NATUREZA)
print('  configuração efetiva:', _reg['parametros_de_treinamento'])
print('  conjuntos/versões de dados:', _cfg.get('versoes_dados'))
print('  filtro lexical / léxico:', _ec.get('filtro_lexical'), '/', _ec.get('lexico'))
print('  marcadores ausentes (fallback h_cls):', (_trn.get('descricao') or {}).get('marcadores_ausentes'))

### 8.2 Comparação justa no DEV (universo comum)

O Baseline pontua todos os candidatos do DEV; o Two-Stage compõe seus estágios sobre o mesmo DEV; a Pair-Aware (D10) também pontua todos os candidatos do **DEV completo**. A comparação só é feita entre sidecars com o **mesmo `y_true`**; qualquer diferença de universo interrompe a tabela. Execuções que não estiverem no Drive/disco local são listadas como não encontradas — **nenhum valor é inventado**. Ajuste os nomes abaixo para as execuções do Baseline e do Two-Stage que você tem. Só DEV; o TEST não é lido.

In [ ]:
from reclin.experimento import persistencia

NOMES_COMPARACAO = [                 # ajuste aos nomes reais das pastas em MyDrive/RECLin-PT/execucoes
    'pairaware_biobertpt_seed42',          # referência direta: Pair-Aware linear com BioBERTpt-all
    'baseline_biobertpt_seed42',
    'twostage_biobertpt_seed42',
    EXECUCAO_SELECIONADA,
]
try:
    _cmp = comparacao_dev.carregar_e_comparar([RAIZ_LOCAL, persistencia.raiz_das_execucoes(RAIZ_DRIVE)], NOMES_COMPARACAO)
    print(comparacao_dev.formatar(_cmp))
except comparacao_dev.ErroComparacao as erro:
    print('COMPARAÇÃO RECUSADA:', erro)

## 9. Curvas de treino/validação

De `dev_history`: loss de treino vs. validação (overfitting) e F1 no dev por época (seleção de época). Na Pair-Aware, `dev_macro_f1` é o do DEV completo, o critério de escolha da época.

In [ ]:
import json
import matplotlib.pyplot as plt
PASTA = apresentacao.pasta_local_da_execucao(CTX)
r = json.load(open(PASTA / 'treino.json', encoding='utf-8'))
h = r['dev_history']
ep = [e['epoch'] for e in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [e['train_loss'] for e in h], 'o-', label='train_loss')
ax[0].plot(ep, [e.get('dev_loss') for e in h], 's-', label='dev_loss')
ax[0].set_xlabel('época'); ax[0].set_ylabel('loss'); ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, [e['dev_macro_f1'] for e in h], 'o-', label='dev_macro_f1')
ax[1].plot(ep, [e['dev_negation_of_f1'] for e in h], 's-', label='dev_negation_of_f1')
ax[1].set_xlabel('época'); ax[1].set_ylabel('F1'); ax[1].set_title('F1 no dev'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 10. Salvar resultados + predições no GitHub

Forçamos o `add` (`-f`) de cinco arquivos **da execução selecionada na seção 8** (nenhuma outra), em `codigo/resultados/execucoes/<nome>/`: `config.json`, `treino.json`, `experimento.json`, `predicoes_dev.json` (predições do dev na melhor época) e `metricas.json` (**só DEV**). Depois `commit`, `pull --rebase --autostash` e `push`, como no legado.

Nunca vão para o GitHub: `ultimo.pt`, checkpoints, pesos, logs nem artefatos do TEST. O envio exige execução concluída, `metricas.json` sem a chave `test` e predições sem texto clínico, e vai para uma **branch própria** (`main` e `master` são recusadas; o merge para a main é manual, fora do notebook).

**Nada é enviado até você mudar `PUBLICAR_NO_GITHUB` para `True`.** Antes, a célula mostra exatamente o que seria enviado.

In [ ]:
from reclin.experimento import versionamento

PUBLICAR_NO_GITHUB = False          # True = confirmo enviar estes arquivos do DEV ao GitHub
BRANCH_RESULTADOS = 'resultados-colab'   # fixa (main e master são recusadas)

if 'EXECUCAO_SELECIONADA' not in globals():
    raise SystemExit('Rode a seção 8 antes: ela seleciona a execução a enviar.')
CTX = _ctx(execucao=EXECUCAO_SELECIONADA)
PASTA = apresentacao.pasta_local_da_execucao(CTX)
print('Execução a enviar:', EXECUCAO_SELECIONADA, '| natureza:', NATUREZA)
try:
    versionamento.conferir(PASTA, repositorio=f'angeloalsf/{REPO_NAME}', branch=BRANCH_RESULTADOS, sha_codigo=SHA_CODIGO)
    if not PUBLICAR_NO_GITHUB:
        print('\nNada foi enviado. Confira acima; para enviar, ponha PUBLICAR_NO_GITHUB = True e rode de novo.')
    else:
        _cred = segredos.Credencial('GITHUB_PAT', userdata.get('GITHUB_PAT'), 'Colab Secrets')
        try:
            RES_GIT = versionamento.salvar(
                PASTA, repositorio=f'angeloalsf/{REPO_NAME}', branch=BRANCH_RESULTADOS, sha_codigo=SHA_CODIGO,
                clone='/content/reclin_versionamento', nome_git=GIT_USER_NAME, email_git=GIT_USER_EMAIL, credencial=_cred)
            print('Push concluído (se houve mudanças).')
        except versionamento.ErroVersionamento as erro:
            print('ERRO:', segredos.limpar(str(erro), _cred))
        finally:
            _cred = None
except versionamento.ErroVersionamento as erro:
    print('ERRO:', erro)

## 11. Backup privado do melhor modelo no HF Hub (confirmação explícita)

Publica a execução selecionada na seção 8 (`melhor_modelo/` + manifesto com SHA-256, **sem** `ultimo.pt`) no repositório **privado** `HF_BACKUP_REPO` (seção 6), num prefixo próprio da execução; nada de execuções anteriores é sobrescrito. Use também como **rede de segurança** se a rede cair no meio de um envio: rode de novo.

A célula primeiro mostra o plano (o que seria enviado) e **só envia** com `PUBLICAR_NO_HF = True`. O `HF_TOKEN` é lido dos Secrets **só durante o envio** e removido do ambiente ao terminar.

In [ ]:
PUBLICAR_NO_HF = False              # True = confirmo publicar a execução no repositório PRIVADO HF_BACKUP_REPO

if 'EXECUCAO_SELECIONADA' not in globals():
    raise SystemExit('Rode a seção 8 antes: ela seleciona a execução a enviar.')
CTX = _ctx(execucao=EXECUCAO_SELECIONADA, repositorio_hf=HF_BACKUP_REPO)
CTX.confirmacao = operacoes.frase_de_publicacao(CTX.execucao, HF_BACKUP_REPO) if PUBLICAR_NO_HF else ''
print('Execução a publicar:', EXECUCAO_SELECIONADA, '| natureza:', NATUREZA)
operacoes.executar(operacoes.OP_PUBLICACAO, CTX)             # plano: nada é enviado
if not PUBLICAR_NO_HF:
    print('\nNada foi enviado. Para publicar, ponha PUBLICAR_NO_HF = True e rode de novo.')
else:
    try:
        with apresentacao.segredo_no_ambiente(segredos.HUGGING_FACE):
            RES_PUBLICACAO = operacoes.executar(operacoes.OP_PUBLICAR, CTX)
        print('best_model em https://huggingface.co/' + HF_BACKUP_REPO)
    except segredos.ErroSegredo as erro:
        print('ERRO:', erro)
        print(apresentacao.instrucoes_secret(segredos.HUGGING_FACE))